number of paths at different lengths (no loop)

In [10]:
numsw = 80
graphfile_list = ["dring_80_64.edgelist","rrg_80_64.edgelist","df_p40_a2_h19.edgelist"]

In [11]:
def compute_distinct_links_upto_k(link, max_length):
    from collections import defaultdict

    num_nodes = len(link)
    adj_list = {i: [j for j in range(num_nodes) if link[i][j] == 1] for i in range(num_nodes)}
    pair_links = defaultdict(set)

    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            stack = [(i, [i], {i})]
            while stack:
                current, path, visited = stack.pop()
                if len(path) - 1 > max_length:
                    continue
                if current == j and len(path) > 1:
                    for k in range(len(path) - 1):
                        a, b = path[k], path[k + 1]
                        pair_links[(i, j)].add(tuple(sorted((a, b))))
                        pair_links[(j, i)].add(tuple(sorted((a, b))))
                    continue
                for neighbor in adj_list[current]:
                    if neighbor not in visited:
                        stack.append((neighbor, path + [neighbor], visited | {neighbor}))
    return pair_links


In [12]:
import pickle

datadict = dict()
for gfile in graphfile_list:
    print(f"Processing {gfile}")
    datadict[gfile] = list()
    graphfile = f"/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/evaltopologyfiles/{gfile}"
    # read graphfile
    link = list()
    for i in range(numsw):
        link.append(list())
        for j in range(numsw):
            link[i].append(0)
    with open(graphfile,'r') as f:
        lines = f.readlines()
        for line in lines:
            tokens = line.split("->")
            fromsw = int(tokens[0])
            tosw = int(tokens[1])
            link[fromsw][tosw] = 1
            link[tosw][fromsw] = 1

    link_counts_k1 = compute_distinct_links_upto_k(link, max_length=2)
    print("k=1 done")
    link_counts_k2 = compute_distinct_links_upto_k(link, max_length=2)
    print("k=2 done")
    link_counts_k3 = compute_distinct_links_upto_k(link, max_length=3)
    print("k=3 done")

    datadict[gfile] = [link_counts_k1, link_counts_k2, link_counts_k3]

with open('num_path.pkl', 'wb') as f:
    pickle.dump(datadict, f)

Processing dring_80_64.edgelist
k=1 done
k=2 done
k=3 done
Processing rrg_80_64.edgelist
k=1 done
k=2 done
k=3 done
Processing df_p40_a2_h19.edgelist
k=1 done
k=2 done
k=3 done


archive

In [ ]:
numsw = 80
graphfile_list = ["dring_80_64.edgelist","rrg_80_64.edgelist","df_p40_a2_h19.edgelist"]
# for leaf-spine, only #spine amount of length-2 paths

In [ ]:
# import numpy as np

# AnnC: issue here: have loops
# def count_paths(link, max_length=4):
#     # Convert to NumPy array for matrix operations
#     adj = np.array(link)
#     num_nodes = adj.shape[0]

#     # Dictionary to store path counts for each length
#     path_counts = {length: None for length in range(1, max_length + 1)}

#     # Initialize power matrix
#     power = adj.copy()

#     for length in range(1, max_length + 1):
#         path_counts[length] = power.copy()
#         power = power @ adj  # Matrix multiplication

#     return path_counts

In [ ]:
# # AnnC: issue: too slow
# def count_simple_paths(link, max_length=4):
#     num_nodes = len(link)
#     path_counts = {length: [[0]*num_nodes for _ in range(num_nodes)] for length in range(1, max_length+1)}

#     def dfs(start, current, visited, depth):
#         if depth > max_length:
#             return
#         if depth > 0:
#             path_counts[depth][start][current] += 1
#         for neighbor in range(num_nodes):
#             if link[current][neighbor] == 1 and neighbor not in visited:
#                 dfs(start, neighbor, visited | {neighbor}, depth + 1)

#     for node in range(num_nodes):
#         dfs(node, node, {node}, 0)

#     return path_counts


In [14]:
from collections import defaultdict

def count_simple_paths(link, max_length=4):
    num_nodes = len(link)

    # ✅ Convert adjacency matrix to adjacency list
    adj_list = {i: [j for j in range(num_nodes) if link[i][j] == 1] for i in range(num_nodes)}

    # ✅ Initialize path count storage
    path_counts = {length: [[0]*num_nodes for _ in range(num_nodes)] for length in range(1, max_length+1)}

    # ✅ Memoization cache for visited sets (optional, useful for larger graphs)
    visited_cache = defaultdict(set)

    def dfs(start, current, visited, depth):
        if depth > max_length:
            return

        if depth > 0:
            path_counts[depth][start][current] += 1

        for neighbor in adj_list[current]:
            if neighbor in visited:
                continue  # ✅ Early pruning: skip loops
            dfs(start, neighbor, visited | {neighbor}, depth + 1)

    # ✅ Run DFS from each node
    for node in range(num_nodes):
        dfs(node, node, {node}, 0)

    return path_counts


In [13]:
for gfile in graphfile_list:
    graphfile = f"/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/evaltopologyfiles/{gfile}"
    # read graphfile
    link = list()
    for i in range(numsw):
        link.append(list())
        for j in range(numsw):
            link[i].append(0)
    with open(graphfile,'r') as f:
        lines = f.readlines()
        for line in lines:
            tokens = line.split("->")
            fromsw = int(tokens[0])
            tosw = int(tokens[1])
            link[fromsw][tosw] = 1
            link[tosw][fromsw] = 1

    # Suppose link is already populated as a 2D list
    # path_counts = count_paths(link)
    path_counts = count_simple_paths(link, max_length=4)

    # To get number of paths of length 3 from node 2 to node 5:
    # print(f"Paths of length 3 from 2 to 5: {path_counts[3][2][5]}")
    print(f"Graph: {gfile}")
    for i in range(1,5):
        sumnumpath = 0
        countnumpath = 0
        for fromsw in range(numsw):
            for tosw in range(numsw):
                if fromsw!=tosw and path_counts[i][fromsw][tosw]>0:
                    sumnumpath += path_counts[i][fromsw][tosw]
                    countnumpath += 1
        print(f"Length {i} - Avg: {sumnumpath/countnumpath if countnumpath > 0 else 0}, Sum: {sumnumpath}, Count: {countnumpath}")

Graph: dring_80_64.edgelist
Length 1 - Avg: 1.0, Sum: 2132, Count: 2132
Length 2 - Avg: 11.599660729431722, Sum: 54704, Count: 4716
Length 3 - Avg: 218.6772151898734, Sum: 1382040, Count: 6320
Length 4 - Avg: 5409.592405063291, Sum: 34188624, Count: 6320
Graph: rrg_80_64.edgelist
Length 1 - Avg: 1.0, Sum: 2048, Count: 2048
Length 2 - Avg: 7.974683544303797, Sum: 50400, Count: 6320
Length 3 - Avg: 193.78101265822784, Sum: 1224696, Count: 6320
Length 4 - Avg: 4651.079113924051, Sum: 29394820, Count: 6320
Graph: df_p40_a2_h19.edgelist
Length 1 - Avg: 1.0, Sum: 1560, Count: 1560
Length 2 - Avg: 6.724137931034483, Sum: 29640, Count: 4408
Length 3 - Avg: 92.6923076923077, Sum: 549480, Count: 5928
Length 4 - Avg: 1620.1518987341772, Sum: 9727392, Count: 6004


In [15]:
numsw = 80
graphfile_list = ["df_p46_a4_h4.edgelist"]

In [18]:
for gfile in graphfile_list:
    graphfile = f"/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/evaltopologyfiles/{gfile}"
    # read graphfile
    link = list()
    for i in range(numsw):
        link.append(list())
        for j in range(numsw):
            link[i].append(0)
    with open(graphfile,'r') as f:
        lines = f.readlines()
        for line in lines:
            tokens = line.split("->")
            fromsw = int(tokens[0])
            tosw = int(tokens[1])
            link[fromsw][tosw] = 1
            link[tosw][fromsw] = 1

    # Suppose link is already populated as a 2D list
    # path_counts = count_paths(link)
    path_counts = count_simple_paths(link, max_length=5)

    # To get number of paths of length 3 from node 2 to node 5:
    # print(f"Paths of length 3 from 2 to 5: {path_counts[3][2][5]}")
    print(f"Graph: {gfile}")
    for i in range(1,6):
        sumnumpath = 0
        countnumpath = 0
        for fromsw in range(numsw):
            for tosw in range(numsw):
                if fromsw!=tosw and path_counts[i][fromsw][tosw]>0:
                    sumnumpath += path_counts[i][fromsw][tosw]
                    countnumpath += 1
        print(f"Length {i} - Avg: {sumnumpath/countnumpath if countnumpath > 0 else 0}, Sum: {sumnumpath}, Count: {countnumpath}")

Graph: df_p46_a4_h4.edgelist
Length 1 - Avg: 1.0, Sum: 476, Count: 476
Length 2 - Avg: 1.3864077669902912, Sum: 2856, Count: 2060
Length 3 - Avg: 3.6189640035118527, Sum: 16488, Count: 4556
Length 4 - Avg: 20.924460431654676, Sum: 93072, Count: 4448
Length 5 - Avg: 115.28007023705004, Sum: 525216, Count: 4556
